# 🥉 Bronze | MedDra Reacoes

Raw Data
as is production, low quality

In [18]:
%run ../config/bootstrap.py

import pandas as pd 
from pathlib import Path
from utils import get_project_root 
project_root = get_project_root() 


# df 

In [19]:
file_path = Path(project_root) / "data/01_bronze/meddra/MedDRA_28_1_Brazilian_Portuguese/version_report_28_1_Brazilian_Portuguese.xlsx"
xls = pd.ExcelFile(file_path)  # você só "abre" o arquivo
print(xls.sheet_names)

['Resumo', 'Novos LLTs, incluindo novos PTs', 'Novos PTs', 'Promoções', 'Rebaixamentos', 'LLTs sob diferentes PTs', 'Alterações de SOC primário de L', 'Alterações de SOC primário do P', 'Alterações de nome de termo do ', 'Trocas de código do MedDRA', 'Alterações de vigência', 'Alterações complexas', 'Alterações de SMQ', 'Alterações de PT em SMQs', 'Alterações de LLT em SMQs']


## Resumo

In [20]:
r = pd.read_excel(file_path, sheet_name="Resumo")
r.head()

c:\Users\silma\Projetos\vigimed\.venv\lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,Título do relatório,Total de termos únicos entre as versões selecionadas,Relatório selecionado?
0,"Novos LLTs, incluindo novos PTs",697.0,Sim
1,Novos PTs,249.0,Sim
2,Promoções,14.0,Sim
3,Rebaixamentos,20.0,Sim
4,LLTs sob diferentes PTs,104.0,Sim


## LLTs sob diferentes PTs

In [17]:
llt_pt = pd.read_excel(file_path, sheet_name="LLTs sob diferentes PTs")
llt_pt.head()

c:\Users\silma\Projetos\vigimed\.venv\lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,Código do LLT,LLT,28.0 Código do PT,28.0 PT,28.1 Código do PT,28.1 PT,28.1 Vigência
0,10001190,Adenocarcinoma da cárdia em estágio 0,10001150,Adenocarcinoma gástrico,10001153,Adenocarcinoma gástrico em estágio 0,ou Vigente
1,10001191,Adenocarcinoma da cárdia em estágio I,10001150,Adenocarcinoma gástrico,10001154,Adenocarcinoma gástrico em estágio I,ou Vigente
2,10001192,Adenocarcinoma da cárdia em estágio II,10001150,Adenocarcinoma gástrico,10001155,Adenocarcinoma gástrico em estágio II,ou Vigente
3,10001193,Adenocarcinoma da cárdia em estágio III,10001150,Adenocarcinoma gástrico,10001156,Adenocarcinoma gástrico em estágio III,ou Vigente
4,10001195,Adenocarcinoma da cárdia em estágio IV com metástases,10001150,Adenocarcinoma gástrico,10092541,Adenocarcinoma gástrico em estágio IV,ou Vigente


# dim_meddra

In [ ]:

from pathlib import Path
import pandas as pd
 
base = project_root / "data/01_bronze/meddra/MedDRA_28_1_Brazilian_Portuguese/ascii-281"


In [22]:

llt_file = base / "llt.asc"
mdhier_file = base / "mdhier.asc"

# conferir existência
assert llt_file.exists(), llt_file
assert mdhier_file.exists(), mdhier_file

In [ ]:
C:\Users\silma\Projetos\vigimed\data\01_bronze\meddra\MedDRA_28_1_Brazilian_Portuguese\ascii-281\hlgt_hlt.asc

In [ ]:
from pathlib import Path
import pandas as pd
 
base = project_root / "data/01_bronze/meddra/MedDRA_28_1_Brazilian_Portuguese/ascii-281"

llt_file = base / "llt.asc"
mdhier_file = base / "mdhier.asc"

# conferir existência
assert llt_file.exists(), llt_file
assert mdhier_file.exists(), mdhier_file

col_llt = [
    "LLT_CODE", "LLT_NAME", "LLT_WHOART", "LLT_HARTS",
    "LLT_COSTART", "LLT_ICD9", "LLT_ICD9CM", "LLT_ICD10",
    "LLT_JART", "PT_CODE", "LLT_CURRENCY"
]
llt = pd.read_table(llt_file, sep=r"\t", names=col_llt, dtype=str, encoding="latin-1")

col_mdhier = [
    "PT_CODE", "HLT_CODE", "HLGT_CODE", "SOC_CODE",
    "PRIMARY_SOC_FLAG", "PT_NAME", "HLT_NAME", "HLGT_NAME",
    "SOC_NAME", "SOC_ABBREV", "PT_SOC_CODE", "PT_SOC_NAME"
]
mdhier = pd.read_table(mdhier_file, sep=r"\t", names=col_mdhier, dtype=str, encoding="latin-1")
mdhier_primary = mdhier[mdhier["PRIMARY_SOC_FLAG"] == "Y"].copy()

dim_soc_llt = (
    llt.merge(mdhier_primary, on="PT_CODE", how="left")
)[[
    "SOC_CODE", "SOC_NAME", "HLGT_CODE", "HLGT_NAME",
    "HLT_CODE", "HLT_NAME", "PT_CODE", "PT_NAME",
    "LLT_CODE", "LLT_NAME", "PRIMARY_SOC_FLAG",
    "SOC_ABBREV", "PT_SOC_CODE", "PT_SOC_NAME",
    "LLT_WHOART", "LLT_HARTS", "LLT_COSTART",
    "LLT_ICD9", "LLT_ICD9CM", "LLT_ICD10", "LLT_JART", "LLT_CURRENCY"
]].drop_duplicates()

out_path = project_root / "data/03_gold/dim_soc_llt/dim_soc_llt.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
#dim_soc_llt.to_parquet(out_path, index=False)

dim_soc_llt.head()

C:\Users\silma\AppData\Local\Temp\ipykernel_40500\1440872664.py:18: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  llt = pd.read_table(llt_file, sep=r"\t", names=col_llt, dtype=str, encoding="latin-1")
C:\Users\silma\AppData\Local\Temp\ipykernel_40500\1440872664.py:25: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  mdhier = pd.read_table(mdhier_file, sep=r"\t", names=col_mdhier, dtype=str, encoding="latin-1")


In [24]:
dim_soc_llt.head()

,SOC_CODE,SOC_NAME,HLGT_CODE,HLGT_NAME,HLT_CODE,HLT_NAME,PT_CODE,PT_NAME,LLT_CODE,LLT_NAME,PRIMARY_SOC_FLAG,SOC_ABBREV,PT_SOC_CODE,PT_SOC_NAME,LLT_WHOART,LLT_HARTS,LLT_COSTART,LLT_ICD9,LLT_ICD9CM,LLT_ICD10,LLT_JART,LLT_CURRENCY
0,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,10000001$Pneumonite de ventilação$10081988$$$$$$$N$$,None,NaN,NaN,NaN,NaN,None,None,None,None,None,None,None,None
1,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,10000002$Deficiência de 11-beta-hidroxilase$10000002$$$$$$$Y$$,None,NaN,NaN,NaN,NaN,None,None,None,None,None,None,None,None
2,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,10000003$Atividade de 11-oxisteroide aum.$10033315$$$$$$$N$$,None,NaN,NaN,NaN,NaN,None,None,None,None,None,None,None,None
3,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,10000004$Atividade aumentada da 11-oxiesteroide$10033315$$$$$$$Y$$,None,NaN,NaN,NaN,NaN,None,None,None,None,None,None,None,None
4,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,10000005$17-cetosteroides na urina$10000005$$$$$$$Y$$,None,NaN,NaN,NaN,NaN,None,None,None,None,None,None,None,None


In [33]:
from pathlib import Path
import pandas as pd

base = Path(project_root / "data/01_bronze/meddra/MedDRA_28_1_Brazilian_Portuguese/ascii-281")

# ------------ Leitura dos arquivos ------------

# llt.asc – 12 campos: 1 LLT_CODE | 2 LLT_NAME | 3 LLT_WHOART | 4 LLT_HARTS
# 5 LLT_COSTART | 6 LLT_ICD9 | 7 LLT_ICD9CM | 8 LLT_ICD10
# 9 PT_CODE | 10 LLT_CURRENCY | 11 LLT_JART | 12 extra vazio
col_llt = [
    "LLT_CODE", "LLT_NAME", "LLT_WHOART", "LLT_HARTS",
    "LLT_COSTART", "LLT_ICD9", "LLT_ICD9CM", "LLT_ICD10",
    "PT_CODE", "LLT_CURRENCY", "LLT_JART", "LLT_EXTRA"
]
llt = pd.read_csv(
    base / "llt.asc",
    sep=r"\$",
    header=None,
    names=col_llt,
    dtype=str,
    encoding="latin-1",
    engine="python",
)

# mdhier.asc – 12 campos padrão
col_mdhier = [
    "PT_CODE", "HLT_CODE", "HLGT_CODE", "SOC_CODE",
    "PRIMARY_SOC_FLAG", "PT_NAME", "HLT_NAME", "HLGT_NAME",
    "SOC_NAME", "SOC_ABBREV", "PT_SOC_CODE", "PT_SOC_NAME"
]
mdhier = pd.read_csv(
    base / "mdhier.asc",
    sep=r"\$",
    header=None,
    names=col_mdhier,
    dtype=str,
    encoding="latin-1",
    engine="python",
)

# ------------ Filtro para SOC primário ------------

mdhier_primary = mdhier[mdhier["PRIMARY_SOC_FLAG"] == "Y"].copy()

# ------------ Join LLT -> PT -> (HLT, HLGT, SOC) ------------

dim_soc_llt = (
    llt.merge(mdhier_primary, on="PT_CODE", how="left")
      .drop(columns=["LLT_EXTRA"])
      .drop_duplicates()
)

# ------------ Ordenar colunas de saída ------------

dim_soc_llt = dim_soc_llt[
    [
        "SOC_CODE", "SOC_NAME", "SOC_ABBREV",
        "HLGT_CODE", "HLGT_NAME",
        "HLT_CODE", "HLT_NAME",
        "PT_SOC_CODE", "PT_SOC_NAME",
        "PT_CODE", "PT_NAME",
        "LLT_CODE", "LLT_NAME",
        "LLT_WHOART", "LLT_HARTS", "LLT_COSTART",
        "LLT_ICD9", "LLT_ICD9CM", "LLT_ICD10", "LLT_JART",
        "LLT_CURRENCY",
        "PRIMARY_SOC_FLAG",
    ]
]

# ------------ Salvar ------------
out_path = project_root / "data/03_gold/dim_soc_llt/dim_soc_llt.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
#dim_soc_llt.to_parquet(out_path, index=False)

print("dim_soc_llt criada em:", out_path)
print("Registros:", len(dim_soc_llt))
print("Exemplo:") 
dim_soc_llt.head()

dim_soc_llt criada em: C:\Users\silma\Projetos\vigimed\data\03_gold\dim_soc_llt\dim_soc_llt.parquet
Registros: 90471
Exemplo:


,SOC_CODE,SOC_NAME,SOC_ABBREV,HLGT_CODE,HLGT_NAME,HLT_CODE,HLT_NAME,PT_SOC_CODE,PT_SOC_NAME,PT_CODE,PT_NAME,LLT_CODE,LLT_NAME,LLT_WHOART,LLT_HARTS,LLT_COSTART,LLT_ICD9,LLT_ICD9CM,LLT_ICD10,LLT_JART,LLT_CURRENCY,PRIMARY_SOC_FLAG
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10000001,Pneumonite de ventilação,10081988,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10000002,Deficiência de 11-beta-hidroxilase,10000002,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10000003,Atividade de 11-oxisteroide aum.,10033315,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10000004,Atividade aumentada da 11-oxiesteroide,10033315,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10000005,17-cetosteroides na urina,10000005,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN


In [34]:
dim_soc_llt.PT_SOC_NAME.value_counts()

Series([], Name: count, dtype: int64)